# Reinforcement Learning and Bandits

Notebook for **Chapter 15** of the *ml* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ml/ch15.html

Tip: change a number, predict the output, then run the cell again.

## 15.1 Multi-Armed Bandits: Exploration versus Exploitation

In [ ]:
import numpy as np
true_p = np.array([0.04, 0.05, 0.06]); T = 5000; K = 3
def run(policy, seed):
    rng = np.random.default_rng(seed)
    a = np.ones(K); b = np.ones(K); n = np.zeros(K); s = np.zeros(K); R = 0; reg = np.zeros(T)
    for t in range(T):
        if   policy == "ab":     arm = t % K
        elif policy == "greedy": arm = t if t < K else int(np.argmax(s/np.maximum(n,1)))
        elif policy == "eps":    arm = t if t < K else (rng.integers(K) if rng.random() < 0.1 else int(np.argmax(s/n)))
        elif policy == "ucb":    arm = t if t < K else int(np.argmax(s/n + np.sqrt(2*np.log(t+1)/n)))
        elif policy == "ts":     arm = int(np.argmax(rng.beta(a, b)))     # Thompson sampling
        r = rng.random() < true_p[arm]
        n[arm] += 1; s[arm] += r; a[arm] += r; b[arm] += 1 - r
        R += true_p.max() - true_p[arm]; reg[t] = R
    return reg

## 15.2 Markov Decision Processes, Rewards and Bellman Equations

## 15.3 Q-learning, SARSA and the Deep-RL Zoo

In [ ]:
def qlearn(seed, episodes=3000, alpha=0.1, eps=0.2, sarsa=False):
    rng = np.random.default_rng(seed); Q = np.zeros((N, N, 4)); rets = []
    for ep in range(episodes):
        s = (0, 0); G = 0
        a = rng.integers(4) if rng.random() < eps else int(np.argmax(Q[s]))
        for t in range(100):
            ns, rw, d = step(s, a); G += rw
            na = rng.integers(4) if rng.random() < eps else int(np.argmax(Q[ns]))
            tgt = rw if d else rw + gamma * (Q[ns][na] if sarsa else Q[ns].max())
            Q[s][a] += alpha * (tgt - Q[s][a]); s, a = ns, na
            if d: break
        rets.append(G)
    return Q, rets

## 15.4 Monte Carlo, TD and Dynamic Programming Side by Side

In [ ]:
import numpy as np
# 5-state random walk: states 1..5, start at 3, step left/right with prob 0.5.
# Left end (0) pays 0, right end (6) pays 1. True value of state i is i/6.
true = np.arange(1, 6) / 6
def episode(rng):
    s, path = 3, []
    while 0 < s < 6:
        path.append(s); s += rng.choice([-1, 1])
    return path, float(s == 6)

def mc(rng, n_ep, alpha):
    V = np.full(7, 0.5); V[0] = V[6] = 0
    for _ in range(n_ep):
        path, G = episode(rng)
        for s in path: V[s] += alpha * (G - V[s])
    return V[1:6]

def td(rng, n_ep, alpha):
    V = np.full(7, 0.5); V[0] = V[6] = 0
    for _ in range(n_ep):
        s = 3
        while 0 < s < 6:
            ns = s + rng.choice([-1, 1]); r = float(ns == 6)
            V[s] += alpha * (r + V[ns] - V[s]); s = ns
    return V[1:6]

for alpha in (0.02, 0.05, 0.1):
    e_mc, e_td = [], []
    for seed in range(200):
        e_mc.append(np.sqrt(np.mean((mc(np.random.default_rng(seed), 50, alpha) - true) ** 2)))
        e_td.append(np.sqrt(np.mean((td(np.random.default_rng(seed), 50, alpha) - true) ** 2)))
    print("alpha %.2f  RMS error after 50 episodes: MC %.3f  TD(0) %.3f" % (alpha, np.mean(e_mc), np.mean(e_td)))

alpha 0.02  RMS error after 50 episodes: MC 0.111  TD(0) 0.129
alpha 0.05  RMS error after 50 episodes: MC 0.116  TD(0) 0.056
alpha 0.10  RMS error after 50 episodes: MC 0.161  TD(0) 0.055


## 15.5 Policy Gradients: Learning the Policy Directly

In [ ]:
import numpy as np
# One state, two actions, softmax policy with logits theta. REINFORCE gradient by hand and by code.
theta = np.array([0.0, 0.0])
pi = np.exp(theta) / np.exp(theta).sum()
print("pi:", pi)
# we took action 1 and got return G = 4
a, G = 1, 4.0
grad_logp = -pi.copy(); grad_logp[a] += 1       # d log pi(a) / d theta = onehot(a) - pi
print("grad log pi(a=1):", grad_logp)
alpha = 0.1
theta_new = theta + alpha * G * grad_logp
pi_new = np.exp(theta_new) / np.exp(theta_new).sum()
print("theta after update:", theta_new)
print("pi after update:", np.round(pi_new, 4))
# with a baseline b = 3 the push is smaller
b = 3.0
theta_b = theta + alpha * (G - b) * grad_logp
pi_b = np.exp(theta_b) / np.exp(theta_b).sum()
print("with baseline 3, pi:", np.round(pi_b, 4))

pi: [0.5 0.5]
grad log pi(a=1): [-0.5  0.5]
theta after update: [-0.2  0.2]
pi after update: [0.4013 0.5987]
with baseline 3, pi: [0.475 0.525]


In [ ]:
import numpy as np
N, gamma = 4, 0.9
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # up, down, left, right
def step(s, a):
    r, c = s[0] + MOVES[a][0], s[1] + MOVES[a][1]
    if not (0 <= r < N and 0 <= c < N): r, c = s
    if (r, c) == (3, 3): return (r, c), 10.0, True
    if (r, c) == (1, 1): return (r, c), -10.0, True
    return (r, c), -1.0, False

def softmax(z):
    e = np.exp(z - z.max()); return e / e.sum()

def reinforce(seed, episodes=3000, lr=0.05, baseline=False):
    rng = np.random.default_rng(seed)
    theta = np.zeros((N, N, 4)); b = np.zeros((N, N))   # b: running mean return per state
    rets = []
    for ep in range(episodes):
        s, traj = (0, 0), []
        for t in range(60):
            p = softmax(theta[s]); a = rng.choice(4, p=p)
            ns, rw, d = step(s, a); traj.append((s, a, rw)); s = ns
            if d: break
        G, Gs = 0.0, []
        for (_, _, rw) in reversed(traj):
            G = rw + gamma * G; Gs.append(G)
        Gs.reverse()
        for (s_, a_, _), G_ in zip(traj, Gs):
            adv = G_ - (b[s_] if baseline else 0.0)
            g = -softmax(theta[s_]); g[a_] += 1
            theta[s_] += lr * adv * g
            if baseline: b[s_] += 0.1 * (G_ - b[s_])
        rets.append(sum(rw for _, _, rw in traj))
    return theta, np.array(rets)

for base in (False, True):
    last = []
    for seed in range(10):
        _, r = reinforce(seed, baseline=base); last.append(r[-200:].mean())
    print("baseline   " if base else "no baseline", "per-seed mean return, last 200 episodes:", np.round(last, 1))
    print("            mean %.2f, std across seeds %.2f" % (np.mean(last), np.std(last)))

no baseline per-seed mean return, last 200 episodes: [  5.    4.9   5.    5.  -60.    5.  -11.    5.    4.9   5. ]
            mean -3.13, std across seeds 19.54
baseline    per-seed mean return, last 200 episodes: [5. 5. 5. 5. 5. 5. 5. 5. 5. 5.]
            mean 4.99, std across seeds 0.01


## 15.6 DQN Mechanics and Choosing Among Bandit Policies

In [ ]:
import numpy as np
# DQN target arithmetic for one stored transition (s, a, r, s').
gamma = 0.9
q_online_s = np.array([1.0, 2.5])          # online network Q(s, .) for 2 actions
a, r = 1, -1.0
q_target_next = np.array([3.0, 5.0])       # frozen target network Q_target(s', .)
q_online_next = np.array([4.0, 2.0])       # online network Q(s', .)
y_dqn = r + gamma * q_target_next.max()
print("DQN target y:", y_dqn)
a_star = q_online_next.argmax()            # Double DQN: online net picks, target net scores
y_ddqn = r + gamma * q_target_next[a_star]
print("Double DQN target y:", round(y_ddqn, 2))
loss = (y_dqn - q_online_s[a]) ** 2
print("squared TD error for DQN:", loss)
# terminal transition: no bootstrap
print("terminal transition, reward 10, target:", 10.0)

DQN target y: 3.5
Double DQN target y: 1.7
squared TD error for DQN: 1.0
terminal transition, reward 10, target: 10.0


In [ ]:
import numpy as np
p = np.array([0.20, 0.25, 0.30]); T = 2000
def run(policy, seed):
    rng = np.random.default_rng(seed)
    a_ = np.ones(3); b_ = np.ones(3); n = np.zeros(3); w = np.zeros(3); regret = 0.0
    for t in range(T):
        if policy == "uniform":
            arm = rng.integers(3)
        elif policy == "eps-greedy":
            arm = rng.integers(3) if (rng.random() < 0.1 or n.min() == 0) else int(np.argmax(w / n))
        elif policy == "UCB":
            arm = int(np.argmin(n)) if n.min() == 0 else int(np.argmax(w / n + np.sqrt(2 * np.log(t + 1) / n)))
        else:
            arm = int(np.argmax(rng.beta(a_, b_)))
        x = rng.random() < p[arm]
        n[arm] += 1; w[arm] += x; a_[arm] += x; b_[arm] += 1 - x
        regret += p.max() - p[arm]
    return regret
for pol in ["uniform", "eps-greedy", "UCB", "Thompson"]:
    r = [run(pol, s) for s in range(200)]
    print("%-10s mean regret after %d rounds: %6.1f (std %.1f)" % (pol, T, np.mean(r), np.std(r)))

uniform    mean regret after 2000 rounds:   99.9 (std 1.7)
eps-greedy mean regret after 2000 rounds:   38.8 (std 38.4)
UCB        mean regret after 2000 rounds:   60.9 (std 7.3)
Thompson   mean regret after 2000 rounds:   28.6 (std 19.5)


## Chapter recap

## Practice set

**1. (easy)** A page has 7 clicks in 50 shows. Write its Beta belief and mean (start from `Beta(1,1)`). **Answer:** `α = 1 + 7 = 8`, `β = 1 + 43 = 44`; mean `8/52 = 0.154`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ml/ch15.html